<a href="https://colab.research.google.com/github/MahfuzAhmed28/Glaucoma_detection_customized_cbam/blob/main/CustomCBAM%26SplitIndices.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Split Indices**

In [3]:
import os
import pandas as pd
from sklearn.model_selection import train_test_split

SEED = 1
DATADIR = r"/content/drive/MyDrive/ACRIMA"
CATEGORIES = ["normal", "glaucoma"]

filepaths = []
labels = []

# Collecting file and label
for category in CATEGORIES:
    path = os.path.join(DATADIR, category)
    class_num = CATEGORIES.index(category)
    for img in os.listdir(path):
        img_path = os.path.join(path, img)
        if os.path.isfile(img_path):
            filepaths.append(img)
            labels.append(class_num)

# Spliting the data for training and testing
files_train, files_test, y_train, y_test = train_test_split(
    filepaths,
    labels,
    test_size=0.20,
    random_state=SEED,
    stratify=labels
)

# Save in CSV file
split_df = pd.concat([
    pd.DataFrame({"filename": files_train, "class": y_train, "split": "training"}),
    pd.DataFrame({"filename": files_test, "class": y_test, "split": "testing"})
], ignore_index=True)

split_df.to_csv("ACRIMA_split_indices_seed1.csv", index=False)

print("Split indices saved successfully as 'ACRIMA_split_indices_seed1.csv'!")
print(split_df.groupby(["split", "class"]).size())

Split indices saved successfully as 'ACRIMA_split_indices_seed1.csv'!
split     class
testing   0         62
          1         79
training  0        247
          1        317
dtype: int64


**Custom CBAM**

In [ ]:
import tensorflow as tf
from tensorflow.keras.layers import Layer, Conv2D, GlobalAveragePooling2D, GlobalMaxPooling2D, Reshape, Dense, multiply, Add, Activation, Concatenate
from tensorflow.keras import backend as K
from tensorflow.keras.utils import register_keras_serializable

@register_keras_serializable()
class CBAM(Layer):
    def __init__(self, filters, ratio=8, **kwargs):
        super(CBAM, self).__init__(**kwargs)
        self.filters = filters
        self.ratio = ratio

    def build(self, input_shape):
        self.shared_dense_one = Dense(self.filters // self.ratio,
                                      activation='relu',
                                      kernel_initializer='he_normal',
                                      use_bias=True)
        self.shared_dense_two = Dense(self.filters,
                                      kernel_initializer='he_normal',
                                      use_bias=True)
        self.spatial_conv = Conv2D(filters=1,
                                   kernel_size=7,
                                   strides=1,
                                   padding='same',
                                   activation='sigmoid',
                                   kernel_initializer='he_normal',
                                   use_bias=False)

    def call(self, inputs):
        # Channel Attention
        avg_pool = GlobalAveragePooling2D()(inputs)
        avg_pool = Reshape((1, 1, self.filters))(avg_pool)
        avg_pool = self.shared_dense_one(avg_pool)
        avg_pool = self.shared_dense_two(avg_pool)

        max_pool = GlobalMaxPooling2D()(inputs)
        max_pool = Reshape((1, 1, self.filters))(max_pool)
        max_pool = self.shared_dense_one(max_pool)
        max_pool = self.shared_dense_two(max_pool)

        channel_attention = Activation('sigmoid')(Add()([avg_pool, max_pool]))
        x = multiply([inputs, channel_attention])

        # Spatial Attention
        avg_pool = tf.reduce_mean(x, axis=-1, keepdims=True)
        max_pool = tf.reduce_max(x, axis=-1, keepdims=True)
        std_pool = tf.math.sqrt(tf.math.reduce_variance(x, axis=-1, keepdims=True) + 1e-6)
        concat = Concatenate(axis=-1)([avg_pool, max_pool,std_pool])
        spatial_attention = self.spatial_conv(concat)
        refined = multiply([x, spatial_attention])

        return refined

    def get_config(self):
        config = super(CBAM, self).get_config()
        config.update({
            'filters': self.filters,
            'ratio': self.ratio
        })
        return config
